In [54]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain.prompts import HumanMessagePromptTemplate,SystemMessagePromptTemplate,PromptTemplate
from langchain_core.messages import SystemMessage
from langchain_core.pydantic_v1 import BaseModel, Field, validator
from typing import List
from langchain_community.callbacks import get_openai_callback
import PyPDF2
import re




In [50]:
def extract_text(pdf_path: str):
    resume_text = ""
    with open(pdf_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        num_pages = len(pdf_reader.pages)

        for page_num in range(num_pages):
            page = pdf_reader.pages[page_num]
            text = page.extract_text().split("\n")

            # Remove Unicode characters from each line
            cleaned_text = [re.sub(r'[^\x00-\x7F]+', '', line) for line in text]

            # Join the lines into a single string
            cleaned_text_string = '\n'.join(cleaned_text)
            resume_text += cleaned_text_string
        
        return resume_text

In [51]:
from pydantic import BaseModel, EmailStr, HttpUrl
from typing import List, Optional


class Profile(BaseModel):
    name: str = Field( description="Full name of the individual.")
    contact_number: Optional[str] = Field(None, description="Contact phone number.")
    email: Optional[str] = Field(None, description="Email address.")
    summary: Optional[str] = Field(None, description="Brief summary about the individual's professional background.")
    address: Optional[str] = Field(None, description="Home or work address.")

    # Validate the email address
    @validator('email')
    def email_validator(cls, v):
        if v is not None:
            # it should be a valid email adress "characters@characters.characters"
            if not re.match(r"[^@]+@[^@]+\.[^@]+", v):
                raise ValueError("Invalid email address")
        return v

class HardSkills(BaseModel):
    """
    This class represent all the relevant technologies and skills that the individual has. Examples are programming languages, frameworks, tools, etc.
    """
    name: str = Field( description="Name of the skill or technology.")
    years_of_experience: Optional[float] = Field(None, description="Number of years of experience with the skill.")

class SoftSkills(BaseModel):
    """
    This class represent all the relevant soft skills that the individual has. Examples are communication, leadership, teamwork, etc.
    """
    name: str = Field( description="Name of the soft skill.")
    proficiency: Optional[str] = Field(None, description="Level of proficiency in the skill.")


class WorkExperience(BaseModel):
    position_title: str = Field( description="Title of the position held.")
    company_name: str = Field( description="Name of the company where the position was held.")
    start_date: str = Field( description="Start date of the employment in YYYY-MM format.")
    end_date: Optional[str] = Field(None, description="End date of the employment in YYYY-MM format, if applicable.")
    accomplishments: List[str] = Field( description="List of achievements or responsibilities in the position.")

class Education(BaseModel):
    degree: str = Field( description="Name of the degree obtained.")
    institution: str = Field( description="Name of the educational institution.")
    start_date: str = Field( description="Start date of the degree program in YYYY-MM format.")
    end_date: Optional[str] = Field(None, description="End date of the degree program in YYYY-MM format, if applicable.")
    description: Optional[str] = Field(None, description="Brief description of the course or program.")
    thesis_title: Optional[str] = Field(None, description="Title of the thesis, if applicable.")

class Language(BaseModel):
    language: str = Field( description="Name of the language.")
    proficiency: str = Field( description="Level of proficiency in the language.")

class CV(BaseModel):
    profile: Profile = Field( description="Profile information of the individual.")
    experiences: List[WorkExperience] = Field( description="List of work experiences.")
    education: List[Education] = Field( description="Educational background details.")
    hard_skills: List[HardSkills] = Field(description="List of hard skills and years of experience.")
    soft_skills: List[SoftSkills] = Field(None, description="List of soft skills and proficiency levels.")
    languages: Optional[List[Language]] = Field(None, description="List of languages spoken and proficiency levels.")
    personal_links: Optional[List[HttpUrl]] = Field(None, description="List of personal or professional links.")



In [52]:

llm = ChatOpenAI(temperature=0.3,api_key=api_key,model="gpt-3.5-turbo-0125")
output_parser = JsonOutputParser(pydantic_object=CV)

cv_text = extract_text("../data/cv/main.pdf")
system_prompt = PromptTemplate(
    template="Objective:  Parse a text-formatted resume efficiently and extract diverse applicant's data based following these output instructions. \n{format_instructions}",
    input_variables=[],
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)

system_prompt = SystemMessagePromptTemplate(prompt=system_prompt)

human_message_prompt = HumanMessagePromptTemplate.from_template("{text}")

chat_prompt = ChatPromptTemplate.from_messages(
    [
        system_prompt,
        human_message_prompt,
    ],
    
)
chain = chat_prompt | llm
with get_openai_callback() as cb:
    result=chain.invoke({"text": cv_text })
    print(cb)

/Users/pmassaro/miniforge3/envs/jobseeker/lib/python3.11/site-packages/pydantic/json_schema.py:2099: PydanticJsonSchemaWarning: Default value default=PydanticUndefined description='Full name of the individual.' extra={} is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/pmassaro/miniforge3/envs/jobseeker/lib/python3.11/site-packages/pydantic/json_schema.py:2099: PydanticJsonSchemaWarning: Default value description='Contact phone number.' extra={} is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
/Users/pmassaro/miniforge3/envs/jobseeker/lib/python3.11/site-packages/pydantic/json_schema.py:2099: PydanticJsonSchemaWarning: Default value description='Email address.' extra={} is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWa

Tokens Used: 2609
	Prompt Tokens: 1561
	Completion Tokens: 1048
Successful Requests: 1
Total Cost (USD): $0.0023525


In [53]:
#create a json from the result
import json
# convert the content into a dictionary
result_dict = json.loads(result.content)
with open("cv.json", "w") as f:
    json.dump(result_dict, f, indent=4)

In [ ]:
PromptTemplate